# Confirm Patient Finder List

## Design

1. Edit only the **Configuration** cell when database, schema, table names, or column names change.
2. Validate the configured architecture against Snowflake before scanning data.
3. **Step A**, per table and per column, learns which columns and which wording carry amyloid text.
4. **A.8** adds any missed wording to the ATTR list, then **Step B** confirms patients with it.
4. Keep family-history evidence separate from patient-confirming evidence.
5. Read warehouse data only; optional outputs are session `TEMPORARY` tables.

In [ ]:
import re
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# ================================================================
# CONFIGURATION - update this cell when the source architecture changes
# ================================================================
DATABASE = 'QUALDERM_MODMED_NEW'
SCHEMA = 'PUBLIC'
MAX_DETAIL_ROWS_PER_TABLE = 5_000
INCLUDE_FAMILY_HISTORY_IN_CONFIRMED = False
TEMP_OUTPUT_TABLES = {
    'patients': 'ATTR_CONFIRMED_PATIENTS',
    'summary': 'ATTR_CONFIRMED_TABLE_SUMMARY',
    'detail': 'ATTR_CONFIRMED_MATCH_DETAIL',
}

# Each column value is a list of acceptable physical names. The first name is
# from Data Dictionary v1.1; alternatives support common Snowflake naming.
# role: CONFIRMING, FHX_REVIEW, or CONTEXT_ONLY.
TABLE_CONFIG = {
    'CENSUS': { # logical name used in the notebook. You can keep this even if Snowflake’s table name is different.
        'table': 'CENSUS', 'analysis_use': 'need: patient demographics/context', # expected Snowflake table name (CENSUS). Change this if the warehouse table is named something else.
        'enabled': False, 'role': 'CONTEXT_ONLY',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': [], 'code_columns': [],
        'context_columns': ['BirthDate', 'Gender', 'City', 'State', 'FamilyId'],
    },
    'ENCOUNTER_VISIT': {
        'table': 'ENCOUNTER_VISIT', 'analysis_use': 'need: encounter linkage/date; not evidence alone',
        'enabled': False, 'role': 'CONTEXT_ONLY',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': [], 'code_columns': [],
        'context_columns': ['EncounterId/VisitId', 'Encounter/Visit Date'],
    },
    'LAB': {
        'table': 'LAB', 'analysis_use': 'need',
        'enabled': True, 'role': 'CONFIRMING',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['ObservationIdentifier', 'ObservationValue', 'LabResultNote'],
        'code_columns': ['ObservationIdentifier', 'ObservationValue', 'LabResultNote'],
        'context_columns': ['LabId', 'EncounterId/VisitId', 'ObservationDateTime'],
    },
    'SOCIAL_HISTORY': {
        'table': 'SOCIAL_HISTORY', 'analysis_use': 'no need for ATTR detection',
        'enabled': False, 'role': 'CONTEXT_ONLY',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['Value'], 
        'code_columns': [], 
        'context_columns': ['Date'],
    },
    'SURGICAL_HISTORY': {
        'table': 'SURGICAL_HISTORY', 'analysis_use': 'need', 
        'enabled': True, 'role': 'CONFIRMING',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['Source/Category', 'Value', 'SNOMED', 'Secondary SNOMED'],
        'code_columns': ['SNOMED', 'Secondary SNOMED'],
        'context_columns': ['SurgicalHistoryId', 'EncounterId/VisitId', 'Date'],
    },
    'MEDICAL_HISTORY': {
        'table': 'MEDICAL_HISTORY', 'analysis_use': 'need', 
        'enabled': True, 'role': 'CONFIRMING',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['Source/Category', 'Value', 'SNOMED', 'Secondary SNOMED'],
        'code_columns': ['SNOMED', 'Secondary SNOMED', 'Value'],
        'context_columns': ['MedicalHistoryId', 'EncounterId/VisitId', 'Date'],
    },
    'FAMILY_HISTORY': {
        'table': 'FAMILY_HISTORY', 'analysis_use': 'need, but family history is not patient confirmation',
        'enabled': True, 'role': 'FHX_REVIEW',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['SNOMED', 'Condition', 'Status', 'FamilyMember'],
        'code_columns': ['SNOMED', 'Condition'],
        'context_columns': ['FamilyHistoryId', 'EncounterId/VisitId', 'Date'],
    },
    'MEDICATION': {
        'table': 'MEDICATION', 'analysis_use': 'no need in v1.1; optional treatment-support evidence',
        'enabled': False, 'role': 'CONTEXT_ONLY',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['Medication Name', 'NDC Code'], 'code_columns': [],
        'context_columns': ['MedicationId', 'EncounterId/VisitId', 'Prescription Date', 'Date'],
    },
    'CLINICAL_NOTE': {
        'table': 'CLINICAL_NOTE', 'analysis_use': 'need', 
        'enabled': True, 'role': 'CONFIRMING',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': ['NoteType', 'Clinical Note Text'], 'code_columns': ['Clinical Note Text'],
        'context_columns': ['NoteId', 'EncounterId/VisitId', 'Date'],
    },
    'CLAIM': {
        'table': 'CLAIM', 'analysis_use': 'need: diagnosis/procedure/provider specialty/notes',
        'enabled': True, 'role': 'CONFIRMING',
        'patient_id': ['Member/PatientId', 'MEMBER_PATIENT_ID', 'PATIENT_ID'],
        'search_columns': [
            'ProcedureCode', 'ProcedureModifier1', 'ProcedureModifier2', 'ProcedureModifier3',
            'DiagnosisType', 'DiagnosisCode', 'ProviderType', 'SpecialtyCode', 'SpecialtyName',
            'DRGCode', 'OtherDiagnosisCodes9', 'OtherDiagnosisCodes10', 'ClinicalNotes',
        ],
        'code_columns': ['DiagnosisCode', 'OtherDiagnosisCodes9', 'OtherDiagnosisCodes10', 'ClinicalNotes'],
        'context_columns': ['EncounterId/VisitId'],
    },
}

session.sql(f'USE DATABASE "{DATABASE}"').collect()
session.sql(f'USE SCHEMA "{SCHEMA}"').collect()
print('Database/schema:', session.get_current_database(), session.get_current_schema())

## Data-dictionary analysis decisions

- **Used for ATTR evidence:** Lab, Surgical History, Medical History, Clinical Note, and Claim.
- **Review separately:** Family History. A relative's disease must not confirm the patient.
- **Context/linkage only:** Census and Encounter/Visit.
- **Not scanned by default:** Social History and Medication. Medication can be enabled as supporting treatment evidence, but medication alone should not prove diagnosis.

These are analysis choices, not a deletion of source fields. Change `enabled`, `role`, `search_columns`, or `code_columns` in the configuration cell to revise them.

In [ ]:
# Search dictionaries are independent from physical table/column names.
BROAD_AMYLOID_TERMS = [
    'amyloidosis', 'amyloidoses', 'amyloidose', 'amiloidosis', 'amiloid', 'amyloid',
    'cardiac amyloid', 'amyloid cardiomyopathy', 'amyloid heart', 'primary amyloid',
    'secondary amyloid', 'familial amyloid', 'senile amyloid', 'systemic amyloid',
    'hereditary amyloid', 'E85', 'E85.82', 'E8582', 'E85.1', 'E851', 'E85.81',
    'E8581', 'E85.89', 'E8589', '237877004', '16573007', '42295001', '442012008',
]
# 'E85' catches the whole ICD amyloidosis family in free text (E85.0/E85.4/E85.9 etc.),
# which is the point of Step A: see everything, then decide what belongs in Step B.

ATTR_SNOMED = ['237877004',# Wild type ATTR amyloidosis
               '16573007', 
               '42295001', 
               '442012008', 
               '715655000']

ATTR_ICD = ['E85.82',
            'E85.1'
            ]
ATTR_NLP_TERMS = [
    'transthyretin', 'transthyretin amyloidosis', 'amyloidogenic transthyretin',
    'ttr amyloidosis', 'ttr amyloid', 'attr amyloidosis', 'attr amyloid',
    'wild-type attr', 'wild type attr', 'attrwt', 'attr-wt', 'wtattr', 'wt-attr',
    'attrv', 'attr-v', 'vattr', 'v-attr', 'attr-cm', 'attr cm', 'attrcm',
    'hereditary attr', 'hattr', 'h-attr', 'familial amyloid polyneuropathy',
    'senile systemic amyloidosis', 'senile cardiac amyloidosis', 'ttr mutation',
    'ttr gene', 'val122ile', 'v122i', 'thr60ala', 't60a', 'tafamidis',
    'vyndaqel', 'vyndamax', 'patisiran', 'onpattro', 'amvuttra', 'vutrisiran',
    'inotersen', 'tegsedi',
]
ATTR_CODE_TEXT_TERMS = ATTR_ICD + [x.replace('.', '') for x in ATTR_ICD] + ATTR_SNOMED
ATTR_CONFIRMED_TERMS = sorted(set(ATTR_NLP_TERMS + ATTR_CODE_TEXT_TERMS))

print('Broad terms:', len(BROAD_AMYLOID_TERMS))
print('Confirmed ATTR terms:', len(ATTR_CONFIRMED_TERMS))

In [ ]:
def quote_ident(name):
    return '"' + str(name).replace('"', '""') + '"'


def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def canonical_name(name):
    """Compare dictionary names across spaces, slashes, underscores, and case."""
    return re.sub(r'[^A-Z0-9]', '', str(name).upper())


def resolve_name(candidates, available):
    if isinstance(candidates, str):
        candidates = [candidates]
    by_upper = {str(x).upper(): x for x in available}
    by_canonical = {}
    for x in available:
        by_canonical.setdefault(canonical_name(x), x)
    for candidate in candidates:
        if str(candidate).upper() in by_upper:
            return by_upper[str(candidate).upper()]
        if canonical_name(candidate) in by_canonical:
            return by_canonical[canonical_name(candidate)]
    return None


metadata = session.sql(f"""
    SELECT TABLE_NAME, COLUMN_NAME, ORDINAL_POSITION
    FROM {quote_ident(DATABASE)}.INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = {sql_literal(SCHEMA.upper())}
    ORDER BY TABLE_NAME, ORDINAL_POSITION
""").to_pandas()

available_tables = metadata['TABLE_NAME'].drop_duplicates().tolist()
columns_by_table = {
    table: grp.sort_values('ORDINAL_POSITION')['COLUMN_NAME'].tolist()
    for table, grp in metadata.groupby('TABLE_NAME')
}

resolved_config = {}
validation_rows = []
for logical_name, cfg in TABLE_CONFIG.items():
    physical_table = resolve_name(cfg['table'], available_tables)
    available_columns = columns_by_table.get(physical_table, [])
    patient_col = resolve_name(cfg['patient_id'], available_columns)
    search_cols = [resolve_name(x, available_columns) for x in cfg['search_columns']]
    code_cols = [resolve_name(x, available_columns) for x in cfg['code_columns']]
    context_cols = [resolve_name(x, available_columns) for x in cfg.get('context_columns', [])]
    missing_search = [x for x, y in zip(cfg['search_columns'], search_cols) if y is None]

    resolved_config[logical_name] = {
        **cfg,
        'physical_table': physical_table,
        'patient_col': patient_col,
        'resolved_search_columns': list(dict.fromkeys(x for x in search_cols if x)),
        'resolved_code_columns': list(dict.fromkeys(x for x in code_cols if x)),
        'resolved_context_columns': list(dict.fromkeys(x for x in context_cols if x)),
        'ready': bool(physical_table and patient_col and any(search_cols)),
    }
    validation_rows.append({
        'LOGICAL_DATASET': logical_name,
        'ANALYSIS_USE': cfg['analysis_use'],
        'ENABLED': cfg['enabled'],
        'ROLE': cfg['role'],
        'PHYSICAL_TABLE': physical_table,
        'PATIENT_COLUMN': patient_col,
        'SEARCH_COLUMNS_FOUND': ', '.join(x for x in search_cols if x),
        'MISSING_CONFIGURED_SEARCH_COLUMNS': ', '.join(missing_search),
        'READY_TO_SCAN': resolved_config[logical_name]['ready'],
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

blocked = validation_df[
    validation_df['ENABLED'] & ~validation_df['READY_TO_SCAN']
]
if len(blocked):
    print('WARNING: enabled datasets below will be skipped until configuration is fixed:')
    display(blocked)
else:
    print('All enabled evidence datasets are ready.')

In [ ]:
def text_predicate(value_sql, terms, include_attr_token=False):
    text_sql = f"UPPER(COALESCE(TO_VARCHAR({value_sql}), ''))"
    predicates = [f"{text_sql} LIKE {sql_literal('%' + term.upper() + '%')}" for term in terms]
    if include_attr_token:
        predicates.append(
            f"REGEXP_LIKE({text_sql}, {sql_literal('.*(^|[^A-Z0-9])ATTR([^A-Z0-9]|$).*')})"
        )
    return '(' + ' OR '.join(predicates) + ')'


def structured_code_predicate(value_sql):
    text_sql = f"UPPER(TRIM(COALESCE(TO_VARCHAR({value_sql}), '')))"
    normalized_sql = f"REPLACE(REPLACE({text_sql}, '.', ''), ' ', '')"
    icd = [
        f"STARTSWITH({normalized_sql}, {sql_literal(code.replace('.', ''))})"
        for code in ATTR_ICD
    ]
    snomed = [f"{text_sql} = {sql_literal(code)}" for code in ATTR_SNOMED]
    return '(' + ' OR '.join(icd + snomed) + ')'


def fully_qualified_table(table_name):
    return '.'.join(map(quote_ident, [DATABASE, SCHEMA, table_name]))


def build_match_union(logical_name, mode='CONFIRMED'):
    cfg = resolved_config[logical_name]
    if not (cfg['enabled'] and cfg['ready']):
        return None

    table_sql = fully_qualified_table(cfg['physical_table'])
    patient_sql = f"T.{quote_ident(cfg['patient_col'])}"
    code_set = set(cfg['resolved_code_columns'])
    parts = []

    for column in cfg['resolved_search_columns']:
        value_sql = f"T.{quote_ident(column)}"
        terms = BROAD_AMYLOID_TERMS if mode == 'BROAD' else ATTR_CONFIRMED_TERMS
        nlp_sql = text_predicate(value_sql, terms, include_attr_token=(mode == 'CONFIRMED'))
        structured_sql = (
            structured_code_predicate(value_sql)
            if mode == 'CONFIRMED' and column in code_set
            else '(FALSE)'
        )
        match_rule_sql = (
            f"CASE WHEN {structured_sql} THEN 'STRUCTURED_CODE' ELSE 'TEXT' END"
            if mode == 'CONFIRMED'
            else "'BROAD_TEXT'"
        )
        parts.append(f"""
            SELECT
                TRIM(TO_VARCHAR({patient_sql})) AS PATIENT_ID,
                {sql_literal(logical_name)} AS LOGICAL_DATASET,
                {sql_literal(cfg['physical_table'])} AS SOURCE_TABLE,
                {sql_literal(column)} AS MATCH_COLUMN,
                TO_VARCHAR({value_sql}) AS EXACT_COLUMN_VALUE,
                {match_rule_sql} AS MATCH_RULE
            FROM {table_sql} T
            WHERE {patient_sql} IS NOT NULL
              AND ({structured_sql} OR {nlp_sql})
        """)
    return '\nUNION ALL\n'.join(parts)


# ---------- Per-column scan engine: one table, one column at a time ----------
scan_store = {'BROAD': {}, 'CONFIRMED': {}}
DISCOVERY_ANCHORS = ['AMYLOID', 'AMILOID', 'TRANSTHYRETIN', 'TTR', 'E85']
DISCOVERY_WINDOW = 40


def terms_for_mode(mode):
    return BROAD_AMYLOID_TERMS if mode == 'BROAD' else ATTR_CONFIRMED_TERMS


def matched_terms(value, mode):
    text = str(value).upper()
    hits = [t for t in terms_for_mode(mode) if t.upper() in text]
    if mode == 'CONFIRMED' and re.search(r'(^|[^A-Z0-9])ATTR([^A-Z0-9]|$)', text):
        hits.append('ATTR-TOKEN(regex)')
    return hits


def column_predicates(logical_name, column, mode):
    cfg = resolved_config[logical_name]
    value_sql = f'T.{quote_ident(column)}'
    patient_sql = f'T.{quote_ident(cfg["patient_col"])}'
    nlp_sql = text_predicate(
        value_sql, terms_for_mode(mode), include_attr_token=(mode == 'CONFIRMED')
    )
    structured_sql = (
        structured_code_predicate(value_sql)
        if mode == 'CONFIRMED' and column in set(cfg['resolved_code_columns'])
        else '(FALSE)'
    )
    return f'{patient_sql} IS NOT NULL AND ({structured_sql} OR {nlp_sql})', structured_sql


def scan_detail(mode, *logical_names):
    names = logical_names or tuple(scan_store[mode])
    parts = [
        scan_store[mode][n]['detail'] for n in names
        if n in scan_store[mode] and len(scan_store[mode][n]['detail'])
    ]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def term_summary(detail_df):
    rows = []
    if detail_df is not None and len(detail_df):
        for _, record in detail_df.iterrows():
            for term in record['MATCHED_TERMS']:
                rows.append({
                    'TERM': term,
                    'MATCH_COLUMN': record['MATCH_COLUMN'],
                    'PATIENT_ID': record['PATIENT_ID'],
                })
    if not rows:
        return pd.DataFrame()
    return (
        pd.DataFrame(rows)
        .groupby('TERM', as_index=False)
        .agg(
            HIT_ROWS=('PATIENT_ID', 'size'),
            UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'),
            COLUMNS=('MATCH_COLUMN', lambda s: ', '.join(sorted(set(s)))),
        )
        .sort_values('UNIQUE_PATIENTS', ascending=False)
    )


def discover_phrases(detail_df, anchors=None, window=DISCOVERY_WINDOW):
    """Text around amyloid/TTR anchors. Rows where IN_CONFIRMED_TERMS is False are
    real wording that Step B would miss today, i.e. candidates for ATTR_EXTRA_TERMS."""
    if detail_df is None or not len(detail_df):
        return pd.DataFrame()
    anchors = anchors or DISCOVERY_ANCHORS
    rows = []
    for _, record in detail_df.iterrows():
        text = str(record['EXACT_COLUMN_VALUE'])
        upper = text.upper()
        in_confirmed = any(t.upper() in upper for t in ATTR_CONFIRMED_TERMS)
        for anchor in anchors:
            for found in re.finditer(re.escape(anchor), upper):
                start = max(0, found.start() - window)
                end = min(len(text), found.end() + window)
                rows.append({
                    'ANCHOR': anchor,
                    'IN_CONFIRMED_TERMS': in_confirmed,
                    'SNIPPET': ' '.join(text[start:end].split()),
                    'LOGICAL_DATASET': record['LOGICAL_DATASET'],
                    'MATCH_COLUMN': record['MATCH_COLUMN'],
                    'PATIENT_ID': record['PATIENT_ID'],
                })
    if not rows:
        return pd.DataFrame()
    return (
        pd.DataFrame(rows)
        .groupby(['ANCHOR', 'IN_CONFIRMED_TERMS', 'SNIPPET'], as_index=False)
        .agg(
            HIT_ROWS=('PATIENT_ID', 'size'),
            UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'),
            SOURCES=('LOGICAL_DATASET', lambda s: ', '.join(sorted(set(s)))),
        )
        .sort_values(['IN_CONFIRMED_TERMS', 'UNIQUE_PATIENTS'], ascending=[True, False])
    )


def run_per_column_scan(logical_name, mode='BROAD', columns=None, show_rows=True):
    """COUNT one column -> skip if zero -> pull rows, exact values, and matched terms."""
    cfg = resolved_config[logical_name]
    print('=' * 70)
    print(f'{mode} | {logical_name} -> {cfg["physical_table"]} | {cfg["analysis_use"]}')
    print('=' * 70)
    if not cfg['ready']:
        print('Skipped: table or columns did not resolve. Fix the configuration cell.')
        return None

    table_sql = fully_qualified_table(cfg['physical_table'])
    patient_sql = f'T.{quote_ident(cfg["patient_col"])}'
    scan_columns = columns if columns is not None else cfg['resolved_search_columns']
    column_rows, detail_parts, patient_ids = [], [], set()

    for column in scan_columns:
        where_sql, structured_sql = column_predicates(logical_name, column, mode)
        counts = session.sql(f"""
            SELECT COUNT(*) AS MATCH_ROWS,
                   COUNT(DISTINCT TRIM(TO_VARCHAR({patient_sql}))) AS UNIQUE_PATIENTS
            FROM {table_sql} T
            WHERE {where_sql}
        """).collect()[0]
        match_rows = int(counts['MATCH_ROWS'])
        unique_patients = int(counts['UNIQUE_PATIENTS'])
        column_rows.append({
            'LOGICAL_DATASET': logical_name,
            'MATCH_COLUMN': column,
            'MATCH_ROWS': match_rows,
            'UNIQUE_PATIENTS': unique_patients,
        })
        print(f'  {column}: rows={match_rows} | patients={unique_patients}')
        if match_rows == 0:
            continue

        select_sql = ', '.join(
            [f'TRIM(TO_VARCHAR({patient_sql})) AS PATIENT_ID']
            + [
                f'TO_VARCHAR(T.{quote_ident(c)}) AS {quote_ident("CTX_" + str(c))}'
                for c in cfg['resolved_context_columns']
            ]
            + [
                f'{sql_literal(column)} AS MATCH_COLUMN',
                f'TO_VARCHAR(T.{quote_ident(column)}) AS EXACT_COLUMN_VALUE',
                f"CASE WHEN {structured_sql} THEN 'STRUCTURED_CODE' ELSE 'TEXT' END AS MATCH_RULE",
            ]
        )
        detail_df = session.sql(f"""
            SELECT {select_sql}
            FROM {table_sql} T
            WHERE {where_sql}
            LIMIT {int(MAX_DETAIL_ROWS_PER_TABLE)}
        """).to_pandas()
        detail_df.insert(0, 'LOGICAL_DATASET', logical_name)
        detail_df['MATCHED_TERMS'] = [
            matched_terms(v, mode) for v in detail_df['EXACT_COLUMN_VALUE']
        ]
        detail_df['MATCHED_TERMS_TEXT'] = [
            ', '.join(t) for t in detail_df['MATCHED_TERMS']
        ]
        detail_parts.append(detail_df)

        ids_df = session.sql(f"""
            SELECT DISTINCT TRIM(TO_VARCHAR({patient_sql})) AS PATIENT_ID
            FROM {table_sql} T
            WHERE {where_sql}
        """).to_pandas()
        patient_ids |= set(ids_df['PATIENT_ID'].astype(str))
        if show_rows:
            display(detail_df.drop(columns=['MATCHED_TERMS']))

    result = {
        'columns': pd.DataFrame(column_rows),
        'detail': pd.concat(detail_parts, ignore_index=True) if detail_parts else pd.DataFrame(),
        'patients': patient_ids,
        'role': cfg['role'],
        'source_table': cfg['physical_table'],
    }
    scan_store[mode][logical_name] = result

    print(f'\n--- {logical_name}: per-column summary ---')
    display(result['columns'].sort_values('UNIQUE_PATIENTS', ascending=False))
    if len(result['detail']):
        print(f'--- {logical_name}: which terms fired ---')
        display(term_summary(result['detail']))
    print(f'{logical_name}: unique patients = {len(patient_ids)}')
    return result

## Step A - broad amyloidosis explore (vocabulary discovery)

**Purpose:** find out what amyloid wording actually exists in the data, so missed keywords can be added to the Step B ATTR list before confirming patients.

**Per column**, one table at a time:

1. SQL `COUNT` for that column.
2. If zero hits, skip to the next column.
3. If hits, pull rows and show the **full exact value** plus **which broad terms fired**.
4. `discover_phrases` prints the text around `amyloid` / `TTR` / `E85`. Rows with `IN_CONFIRMED_TERMS = False` are wording Step B would miss today.

Run one table, read the output, then move to the next. Section **A.7** rolls everything up and **A.8** is where missed keywords get added.

### A.1 `LAB` - broad scan

Lab notes and observation values often carry diagnosis text and pasted ICD codes, so this table usually shows the widest vocabulary.

In [ ]:
run_per_column_scan('LAB', mode='BROAD')
display(discover_phrases(scan_detail('BROAD', 'LAB')).head(30))

### A.2 `MEDICAL_HISTORY` - broad scan

Check `Value` and `Source/Category` wording, and whether SNOMED columns hold text instead of clean codes.

In [ ]:
run_per_column_scan('MEDICAL_HISTORY', mode='BROAD')
display(discover_phrases(scan_detail('BROAD', 'MEDICAL_HISTORY')).head(30))

### A.3 `SURGICAL_HISTORY` - broad scan

Biopsy and cardiac procedure entries sometimes carry the amyloid finding rather than a diagnosis row.

In [ ]:
run_per_column_scan('SURGICAL_HISTORY', mode='BROAD')
display(discover_phrases(scan_detail('BROAD', 'SURGICAL_HISTORY')).head(30))

### A.4 `CLINICAL_NOTE` - broad scan

Free-text notes are the richest source of new vocabulary. Expect abbreviations and phrasing that no code list contains.

In [ ]:
run_per_column_scan('CLINICAL_NOTE', mode='BROAD')
display(discover_phrases(scan_detail('BROAD', 'CLINICAL_NOTE')).head(50))

### A.5 `CLAIM` - broad scan

Diagnosis code columns are the cleanest signal here. Watch the `E85` family to see which amyloidosis types exist in the population, not only ATTR.

In [ ]:
run_per_column_scan('CLAIM', mode='BROAD')
display(discover_phrases(scan_detail('BROAD', 'CLAIM')).head(30))

### A.6 `FAMILY_HISTORY` - broad scan (review only)

Useful for vocabulary and for hereditary ATTR signals, but a relative's condition never confirms the patient. This dataset stays out of the confirmed union unless `INCLUDE_FAMILY_HISTORY_IN_CONFIRMED` is set to `True`.

In [ ]:
run_per_column_scan('FAMILY_HISTORY', mode='BROAD')
display(discover_phrases(scan_detail('BROAD', 'FAMILY_HISTORY')).head(30))

### A.7 Step A roll-up - which columns, which terms, what is missing

The last table is the important one: wording found in the data that the current ATTR term list does **not** cover. Copy anything genuinely ATTR-related from `SNIPPET` into the next cell.

In [ ]:
broad_detail = scan_detail('BROAD')
print('Step A detail rows collected:', len(broad_detail))

broad_summary = pd.DataFrame([
    {
        'LOGICAL_DATASET': name,
        'SOURCE_TABLE': result['source_table'],
        'ROLE': result['role'],
        'MATCH_ROWS': int(result['columns']['MATCH_ROWS'].sum()) if len(result['columns']) else 0,
        'UNIQUE_PATIENTS': len(result['patients']),
    }
    for name, result in scan_store['BROAD'].items()
])
if len(broad_summary):
    display(broad_summary.sort_values('UNIQUE_PATIENTS', ascending=False))

if len(broad_detail):
    print('=== Which columns held amyloid text ===')
    display(
        broad_detail.groupby(['LOGICAL_DATASET', 'MATCH_COLUMN'], as_index=False)
        .agg(HIT_ROWS=('PATIENT_ID', 'size'), UNIQUE_PATIENTS=('PATIENT_ID', 'nunique'))
        .sort_values('UNIQUE_PATIENTS', ascending=False)
    )

    print('=== Which broad terms fired ===')
    display(term_summary(broad_detail))

    print('=== Wording Step B would MISS today (candidates for ATTR_EXTRA_TERMS) ===')
    broad_phrases = discover_phrases(broad_detail)
    missing_phrases = (
        broad_phrases[~broad_phrases['IN_CONFIRMED_TERMS']]
        if len(broad_phrases) else broad_phrases
    )
    print('Candidate snippets:', len(missing_phrases))
    display(missing_phrases.head(100))

### A.8 Add the missed keywords, then run Step B

This is the hand-off between the two steps. Add wording found in A.7 to `ATTR_EXTRA_TERMS`, run this cell, and every Step B scan below picks it up automatically.

Add only ATTR-specific wording. Generic amyloid text such as `AL amyloidosis` belongs in the broad list, not in ATTR confirmation.

In [ ]:
ATTR_EXTRA_TERMS = [
    # 'ttr cardiac amyloidosis',
    # 'transthyretin cardiomyopathy',
]

ATTR_CONFIRMED_TERMS = sorted(set(ATTR_NLP_TERMS + ATTR_CODE_TEXT_TERMS + ATTR_EXTRA_TERMS))
print('Terms added from Step A:', ATTR_EXTRA_TERMS)
print('Confirmed ATTR terms now:', len(ATTR_CONFIRMED_TERMS))

## Step B - confirmed ATTR

Same per-column pattern as Step A, with the stricter list: ATTR wording, the ICD/SNOMED codes as text, and an `ATTR` token regex. Structured ICD/SNOMED matching runs only on configured `code_columns`; `MATCH_RULE` shows `STRUCTURED_CODE` or `TEXT` for every row.

Run A.8 first so any keyword found in Step A is included here.

### Coding reference

| Phenotype | SNOMED | ICD-10 |
|-----------|--------|--------|
| ATTRwt | `237877004` | `E85.82` |
| ATTR-CM senile cardiac | `16573007` | `E85.82` |
| hATTR-PN / FAP | `42295001` | `E85.1` |
| Broad amyloidogenic TTR | `442012008` | `E85.81` / `E85.89` |

### B.1 `LAB` - per column

In [ ]:
run_per_column_scan('LAB', mode='CONFIRMED')

### B.2 `MEDICAL_HISTORY` - per column

In [ ]:
run_per_column_scan('MEDICAL_HISTORY', mode='CONFIRMED')

### B.3 `SURGICAL_HISTORY` - per column

In [ ]:
run_per_column_scan('SURGICAL_HISTORY', mode='CONFIRMED')

### B.4 `CLINICAL_NOTE` - per column

In [ ]:
run_per_column_scan('CLINICAL_NOTE', mode='CONFIRMED')

### B.5 `CLAIM` - per column

`DiagnosisCode` and the other diagnosis columns are matched as structured ICD codes, so `E85.82` and `E8582` both resolve.

In [ ]:
run_per_column_scan('CLAIM', mode='CONFIRMED')

### B.6 `FAMILY_HISTORY` - per column (family history is not patient confirmation)

In [ ]:
run_per_column_scan('FAMILY_HISTORY', mode='CONFIRMED')

### B.7 Union - confirmed ATTR patient list

Combines the `CONFIRMING` datasets scanned above. Family History is reported next to each patient but only joins the union when `INCLUDE_FAMILY_HISTORY_IN_CONFIRMED` is `True`.

In [ ]:
scanned = scan_store['CONFIRMED']
confirmed_summary = pd.DataFrame([
    {
        'LOGICAL_DATASET': name,
        'SOURCE_TABLE': result['source_table'],
        'ROLE': result['role'],
        'MATCH_ROWS': int(result['columns']['MATCH_ROWS'].sum()) if len(result['columns']) else 0,
        'UNIQUE_PATIENTS': len(result['patients']),
        'IN_UNION': result['role'] == 'CONFIRMING' or INCLUDE_FAMILY_HISTORY_IN_CONFIRMED,
    }
    for name, result in scanned.items()
])
if len(confirmed_summary):
    confirmed_summary = confirmed_summary.sort_values('UNIQUE_PATIENTS', ascending=False)
display(confirmed_summary)

confirming_tables = [n for n, r in scanned.items() if r['role'] == 'CONFIRMING']
fhx_tables = [n for n, r in scanned.items() if r['role'] == 'FHX_REVIEW']
union_tables = confirming_tables + (fhx_tables if INCLUDE_FAMILY_HISTORY_IN_CONFIRMED else [])

confirmed_ids = set()
for name in union_tables:
    confirmed_ids |= scanned[name]['patients']

confirmed_rows = []
for patient_id in sorted(confirmed_ids):
    evidence_sources = [n for n in confirming_tables if patient_id in scanned[n]['patients']]
    family_sources = [n for n in fhx_tables if patient_id in scanned[n]['patients']]
    confirmed_rows.append({
        'PATIENT_ID': patient_id,
        'EVIDENCE_DATASETS': ','.join(evidence_sources),
        'N_EVIDENCE_DATASETS': len(evidence_sources),
        'FAMILY_HISTORY_DATASETS': ','.join(family_sources),
        'FAMILY_HISTORY_INCLUDED': INCLUDE_FAMILY_HISTORY_IN_CONFIRMED,
    })

confirmed_patients = pd.DataFrame(confirmed_rows)
print('Confirmed ATTR candidate patients:', len(confirmed_patients))
display(confirmed_patients)

## Review exact evidence for one patient

Set `REVIEW_PATIENT_ID` and rerun this cell. The result shows the exact source value and the physical column that matched.

In [ ]:
REVIEW_PATIENT_ID = ''  # Example: '12345678'

if REVIEW_PATIENT_ID.strip():
    patient_id = REVIEW_PATIENT_ID.strip()
    review_queries = [
        build_match_union(name, mode='CONFIRMED')
        for name in resolved_config
        if build_match_union(name, mode='CONFIRMED')
    ]
    if review_queries:
        all_matches_sql = '\nUNION ALL\n'.join(review_queries)
        patient_evidence = session.sql(f"""
            SELECT *
            FROM ({all_matches_sql}) M
            WHERE PATIENT_ID = {sql_literal(patient_id)}
            ORDER BY LOGICAL_DATASET, MATCH_COLUMN
        """).to_pandas()
    else:
        patient_evidence = pd.DataFrame()
    display(patient_evidence)
else:
    print('Set REVIEW_PATIENT_ID to review one patient.')

## Optional - write session temporary outputs

Run only when temporary Snowflake tables are useful downstream. Names are configurable in `TEMP_OUTPUT_TABLES`; nothing permanent is created.

In [ ]:
patient_output = confirmed_patients if len(confirmed_patients) else pd.DataFrame({
    'PATIENT_ID': pd.Series(dtype='str'),
    'EVIDENCE_DATASETS': pd.Series(dtype='str'),
    'N_EVIDENCE_DATASETS': pd.Series(dtype='int64'),
    'FAMILY_HISTORY_DATASETS': pd.Series(dtype='str'),
    'FAMILY_HISTORY_INCLUDED': pd.Series(dtype='bool'),
})
summary_output = confirmed_summary if len(confirmed_summary) else pd.DataFrame({
    'LOGICAL_DATASET': pd.Series(dtype='str'),
    'UNIQUE_PATIENTS': pd.Series(dtype='int64'),
})
detail_output = scan_detail('CONFIRMED')
if len(detail_output):
    detail_output = detail_output.drop(columns=['MATCHED_TERMS'])
else:
    detail_output = pd.DataFrame({
        'PATIENT_ID': pd.Series(dtype='str'),
        'LOGICAL_DATASET': pd.Series(dtype='str'),
        'MATCH_COLUMN': pd.Series(dtype='str'),
        'EXACT_COLUMN_VALUE': pd.Series(dtype='str'),
        'MATCHED_TERMS_TEXT': pd.Series(dtype='str'),
        'MATCH_RULE': pd.Series(dtype='str'),
    })

for key, frame in [
    ('patients', patient_output), ('summary', summary_output), ('detail', detail_output)
]:
    session.write_pandas(
        frame,
        TEMP_OUTPUT_TABLES[key],
        auto_create_table=True,
        table_type='temporary',
        overwrite=True,
    )
    print('Wrote TEMPORARY', TEMP_OUTPUT_TABLES[key], '| rows:', len(frame))